# CS781 B1: complete-pass reference and document hierarchy

One independently trained model per prediction. Own Phase1 B1 weights and tokenizer only. This segment runs a full-data reference on GPU0 and a 512-token-window document hierarchy on GPU1. The hierarchy uses two document transformer blocks and a bidirectional GRU. Encoder growth and optional FFN expansion are implemented but are not launched before earlier evidence.

Three-hour training segments save exact resumable state. A budget boundary is **paused**, not converged. Published outputs must be downloaded and verified before another segment continues. No live `/kaggle/working` checkpoint is claimed as a D-drive backup. All experiments are after the original course deadline. Accuracy is measured, never guaranteed.


In [ ]:
import os,sys,time,subprocess,importlib.util
from pathlib import Path
SESSION_STARTED=time.time()
SESSION_END=SESSION_STARTED+5.2*3600
os.environ.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
needed={'ijson':'ijson>=3.2,<4','sklearn':'scikit-learn>=1.4,<2','tokenizers':'tokenizers'}
missing=[p for m,p in needed.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('B1_STUDY_BOOT',{'session_start':SESSION_STARTED,'session_end':SESSION_END})


In [ ]:
import base64,hashlib,io,json,zipfile,shutil
EXPECTED_VENDOR={'phase3_rr/__init__.py': 'f38a2af63c8fe34b0e5fe200c4a9c71fb161c9ad6a37bf46ec7307eb24d00ae6', 'phase3_rr/archive_inputs.py': '900109016e17f4170100f04a630b7259e41eee51442d04d015b40a85d0512391', 'phase3_rr/common.py': 'f808cde35564328d8f377dabb79da21e8ef1001623d9897d7b789fd2251e6535', 'phase3_rr/data.py': '00084b779497b1480c9574583faef1b6700a7bd92c4a1ff2fe5c1d1cd0ffc02a', 'phase3_rr/models.py': '14fc38d447e1fcb8d663d00932b24344c74d7c383cf7b543a8b90cc4e97c4eea', 'phase3_rr/train.py': 'db1532039e21f531b81f15b568a7ab97ce86779df5df4b5b0ff1f4299fbd4f63', 'phase3_rr/vendor/__init__.py': '4e813963bf49a55dffeba982788a12a8d854c7e7a4e6ed19b300b4ea35cef565', 'phase3_rr/vendor/backbones/cs781/__init__.py': '4a3613ae45d376f074874f3924962127b635a12fc6a03c5cf2c2d2e9cc3c1c49', 'phase3_rr/vendor/backbones/cs781/checkpoints.py': 'f70275cc12a1ca3bda33c4103dc2f697b0a374f8dcd5a3f30ab9f9f3f3b08418', 'phase3_rr/vendor/backbones/cs781/cli.py': 'e6695a4ec962094bf8fba96d61e61182f421440af61475644ae5dfe9223bfd8f', 'phase3_rr/vendor/backbones/cs781/cli_b.py': '471b80b2b4485a7bbc13a196fbfa452c7faa536fce85c7f9293742012248aa33', 'phase3_rr/vendor/backbones/cs781/cli_c.py': '0e6b8d4aa1b942aeb214881d1d9074e524ce90dfed641b5438a41601874c3aa6', 'phase3_rr/vendor/backbones/cs781/contracts.py': 'b4d17bf29617b79bff3c3507cf01245dbe16ad550c2c394a15aaddedd0733689', 'phase3_rr/vendor/backbones/cs781/corpus.py': '5706941f90284ba4481524a6ffe93c9f108bfcadcb0845107a107c57891098bf', 'phase3_rr/vendor/backbones/cs781/evaluate.py': '9ca3eb30f79592adfce063cdb47a34cb21a84b33b7406c8e888ce980267dba2a', 'phase3_rr/vendor/backbones/cs781/examples.py': '372e9d40984f66bda6873dbd87382416544eee410984651c27a7d000c38b1eba', 'phase3_rr/vendor/backbones/cs781/examples_alt.py': '7260c5b156dcd046c23545ab851c25014aef76a81ca73b612216de80a4c83052', 'phase3_rr/vendor/backbones/cs781/generate.py': '9a3bc31bd2a5e0e7fdb342e81e0fc20c031beab13711dc3bbf96b5728dcc8d03', 'phase3_rr/vendor/backbones/cs781/generate_b.py': 'e92ed53fcf2f6853eda5cc8a48bdacac0c2b92478e4677d4161dc3b4f6eba6ba', 'phase3_rr/vendor/backbones/cs781/generate_c.py': '43be46a4714d76a835f83a792b72210ceb80b889ce30675e13555243bc7d598e', 'phase3_rr/vendor/backbones/cs781/io.py': '6f7c8510cff6eee2d6cc82dbb77cd0d3d3cd7063670c0da37363ca7d600f45e4', 'phase3_rr/vendor/backbones/cs781/models/__init__.py': '0da8971507f5266ea3b85759054329394979d08d7f8e4896472fd471f78d3d49', 'phase3_rr/vendor/backbones/cs781/models/a.py': 'b5eb47496093388acf0798fa44fe2155116bd30a3fd1a9fbec3f6a71b5dca268', 'phase3_rr/vendor/backbones/cs781/models/b.py': '9919cc6a3ac87fb080acf6ac2be2df84bfb0d1d6d314e53fcf6c977bde14c057', 'phase3_rr/vendor/backbones/cs781/models/c.py': 'd9c16e1b53c23db7fd2a2be3e83f09c30642012fd31a63dc8282b7e39dc98147', 'phase3_rr/vendor/backbones/cs781/own_bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/cs781/prediction_progress.py': '3d35956bdecc1f88890010be3cd15527bd33ce463ff48b2a304db470f0a505c4', 'phase3_rr/vendor/backbones/cs781/prepare.py': 'efb7368de000b0e9f44595072b0f7f146e1809ef6f34e799859b14b33de79f9e', 'phase3_rr/vendor/backbones/cs781/splits.py': '2434ecd26e762919b66d4586ab530615c578d677a385bb1c78eb06f4d2cfee83', 'phase3_rr/vendor/backbones/cs781/submission.py': 'c36de151bccce0f844827c33087ec9c1b3163cb2ed7a32e45d7bf198cad1063d', 'phase3_rr/vendor/backbones/cs781/tokenizer.py': 'e838c764e9dc2848c1ea04f9f51b261aab1afe5607cfe8cf7322ab5984555558', 'phase3_rr/vendor/backbones/cs781/train.py': 'a2367df0b50efb36660b067940356b953340240483feb787d2108ee4258d6588', 'phase3_rr/vendor/backbones/cs781/train_b.py': '3d0bac658a06799ece3b90ca23a53f6a2627749188f79418ef093344bdde0758', 'phase3_rr/vendor/backbones/cs781/train_c.py': '63e88403d9e6fbdd754b635d08f9e38d4b1898ff1373e927723e15e8274d0c4e', 'phase3_rr/vendor/backbones/legalspan/__init__.py': 'c24781d92781386918ca9056ca807b62493db4b5096d078dfcb0b1809b48b37e', 'phase3_rr/vendor/backbones/legalspan/bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/legalspan/data.py': '48b7af4ae83e715210f201777ae4c876876af550a9d1f09b485d29250ba7a06b', 'phase3_rr/vendor/backbones/legalspan/generate.py': '315c7067041e2314df5375a0d0b29899af0972366a21da0a546b43687d2748bd', 'phase3_rr/vendor/backbones/legalspan/model.py': '0423c932eaf22a919139ed433f1a5aee5e8a2a3720edf4e3b9d19ac5023435d0', 'phase3_rr/vendor/backbones/legalspan/pipeline.py': 'c20587337d8ac74b24ec517020d9752ea508f31e4b802e5219080f9728692cbb', 'phase3_rr/vendor/backbones/legalspan/train.py': '86a6d99aba0a153d0b0759662ba4257cd3e7955c0283a2550747f943b29017a2', 'phase3_rr/vendor/evaluate_phase3.py': '5b2babf1cb42a6f211c3264b7dc4be61566cd22df9eb2aae8b609ea0ee2f0b1f', 'phase3_rr/vendor/legal_d/__init__.py': 'fe77b5788e408a4d7c774eccd3cd8b359527cc580ce82eacc38fd5294e6c8989', 'phase3_rr/vendor/legal_d/model.py': '35c3ebc178de3a5797884542bb739d225a3ea7fd32af9774b759187c73c272fc', 'phase3_rr/vendor/SOURCES.md': 'b3cacaa420e9604392614d9e6628cc591b9d9c9f494b97b765b16d61fa12b3f4', 'phase3_rr/vendor/UnicodeData-13.0.0.txt': 'bdbffbbfc8ad4d3a6d01b5891510458f3d36f7170422af4ea2bed3211a73e8bb', 'phase3_rr/vendor/UnicodeData-14.0.0.txt': '36018e68657fdcb3485f636630ffe8c8532e01c977703d2803f5b89d6c5feafb', 'phase4_cjpe/__init__.py': 'f18c25fd56e59b628ab217396751039d3abbbf12ef6b0a935337a9c7b58c8576', 'phase4_cjpe/common.py': '0546d963cdf031283a87ce032076ffc832b7f49f15f6af1b0a9d4fc21589e069', 'phase4_cjpe/data.py': '95e6823bd24a6172ef3d7840112046fc818c7b069fa557638bca61adf9577703', 'phase4_cjpe/evaluate_phase4.py': '16f71e82d66b8b7a9e5289db22f3285edea99a4e6df8e9617b5b9e78699e3766', 'phase4_cjpe/export.py': '4a743510077a1fccd518774df56ecade58a5bcdac39fe69fd533408ad1f45090', 'phase4_cjpe/kaggle_runner.py': '8a8a4f82e083f080b93641eec1381b43720cb8e85582bb2a608e58aaf1eaed77', 'phase4_cjpe/models.py': '2a6fc936dd37531472824414c64bc0e561275f90c9448708e08625cbf46b9562', 'phase4_cjpe/online_teacher.py': 'b4ae71b323ced68019127241e991d87c1a6c0995d773f33a6efdd3c914e0aed4', 'phase4_cjpe/runtime.py': '066e8144daf6f5a1976cd52da1a0892955f6a7460e92a41480b28c65920b4469', 'phase4_cjpe/train.py': '2224e8e18e00f77ec52bda3b6c85a1de3428ec2236582791ca995137333248b9'}
EXPECTED_STUDY={'b1_study/__init__.py': '27aa66216e3a4a6db758b70d85bf6d2005adcceb790da04eb3c41190b56dc157', 'b1_study/data.py': '74f4e92d710de2d56fddf9c8917f8e8fea0c8eb081ba832f69db9227e0e378be', 'b1_study/model.py': 'e593a501d0d56934c0571bc9029fb4aaa47ab99a78c895e90a98756316417aa6', 'b1_study/run.py': '1a3ce9211e7657a6623c2766f95a4675828695a4e5057826167ba70eda28e1f4', 'b1_study/train.py': '2b644ff35086d0efadd614156c6d50f0f006856bbf6b4494b09c5645d1416168'}
matches=[]
for p in Path('/kaggle/input').rglob('SOURCE_MANIFEST.json'):
    try:
        if json.loads(p.read_text())==EXPECTED_VENDOR:matches.append(p)
    except (ValueError,OSError):pass
assert len(matches)==1,'Attach exact recovered course runner dataset.'
CODE=Path('/kaggle/working/b1_study_source');CODE.mkdir(parents=True,exist_ok=True)
for relative,wanted in EXPECTED_VENDOR.items():
    data=(matches[0].parent/relative).read_bytes()
    assert hashlib.sha256(data).hexdigest()==wanted,'Changed vendor source:'+relative
    dest=CODE/relative;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
raw=base64.b64decode('UEsDBBQAAAAIAOeFVl2KRSF7PgAAADwAAAAUAAAAYjFfc3R1ZHkvX19pbml0X18ucHlTUlLyL89TCMhILE5VMFRwMtRRKM7MS89J1c3NT0nNUcjITC1KLErOyExOzFFw9gpwVSguKU2p1FNSUuICAFBLAwQUAAAACADnhVZdKwJLRI8MAACxJAAAEAAAAGIxX3N0dWR5L2RhdGEucHmVGmtz2zbyu38Fms4dyYZhJPd5vrAdt1XvMm2TjuO219FoOBAJSaj4CkH60Vz+++0uABKUKDunZGwRWOwb+6JlUVdNy/5UVXkm9Xf1Npet+PRs01QFq3m7y+Wamb1f4PHMApZdUd8zrlhZnxnoHVfisyT9sxZRWhVFVdqDvK0KmSZIJ2SZ3ArVhkztOPxoedup4/MZb7k9LctM3CW4ErJ015V7hWQbsRGNKFOR6DUHx6dJ04wwXFd7Ucq/RHN2dpaJDVM1CJnITPnZOmQbeSeyZFflWdUhX0Jk8ZdfzYOLMwafJ0+e/ChEzepGVg274bkEzBJke/k98FFmbNtUXc3E247nrKyaAiD+EhlrGy5LWW5ZK+5aFQEaQncr251VMmipLEXaAhcBigSPmiZ+mupWsRjXInEn0q4Vvvdm8dPiu2smszDna5GHiDoBPbIfrl7/zFIQXbHf/724WmgJ4yfExBP2+ur7xRX79g846QXRRrTpjue5H/TEqnYnGqD2Tl2wZbOcrdgGhG1A9ScYOEXxmxGtkPkqDIJVT6f/IHqF6H0vEzcA6LXgFF7wnkDX92AcZAdZuQA+em5QKxrG2AuglGj9kQ21XHLDclH6do19FLPP5+cMMJVV2x9/EWtq0V7cKz9w9M+lEuw3nndi0TRV43uLuxqMhabdCUAjt7IEkyPOsVd4mj75BZrwnWYYZQAiIdO2I79IwF13Vq6Btj4agWDgrbzLW38ABiy+J4q6vb/w2FPEGIRsuQoiXteizHx/oBH0iuDlvY/KeEfrxEpigJA60XtPKprTpvZpu6OiG1QDqOdB/bzMRNnK9IFrwHbayze5TFtcJA6swhqhRHODZgQ5BWmOTLPcr5bnqyPBA61R5NIYU+u5AbwxRKWogdtZFZFRYQLrPt5tODv/UlMEXuW60YaLQYmHiwMn6GNBb8Zecf4sZHNHKSmQRE/QZyD0iMzf0ZFdyLaOQuH6F6BPtM2O/BF94EB+Ci5wAZbzFYtjY9KelHHvgWLAXrDzz7+4GF22CSuVqttsZCrBWBCL1Z+VhC+uJjSL3kAK+ZfEYbmNatEUXUuQ/gEDwfICGFiNOdhhEOthlkDMlwcRwSFuvVjzsNytUP7ZKjgFb3QV8QwUraHEXZp3mcD4cajR/04cpTPkpYPJtprqhxrOUuzdGNwNkGXg5D5hjulnaP00tp5hQlPosmU3nSUA+OQTCtGawnA/IFx/6B3RbC3V6v1BNHZCMbGJwdhwhl8dPmx4xhOQi9d0aOIU8OusjhDgzinkzj0C7eKNMzzzVcD+7i6sYQFNQ3qAbdg139erx/z/2gSk50PQfu56fy74nm9FH5ParrHKC9k7bwhtiQ1tibkwF3Qh9QMI6lV0zXg+AELFYaAw3gfhcV6Ej+eCJ50SmTlD64h44JxAauvmAHfddOIEWtf106orW+RlCRc2xLCxArzEemJjwsPYrM8nWQeJP4W7fVJM9sxhHx7ms/PP3ptK7BaKu+o2abFAU77+FbKC35mlGNIrXA9kN54Dp9WNaHJex/MZRN66Asr3sddymXvGe9ZdthV4+wYcQPLcJkKz/YLNv7CVwAxrAIMXNgwEbBJVDKoPJr2SrGEEgTtfAEVQsY2fGFVIemKlT8hI2C32tJstl/OQfQr2WIE5lrNohR636fLcSAJaBbG9rpRvOzFeAwygXvTKYXXSckdugaKDwyebhqd0ES/YPAKMWiLENNfXXrWNzATIY1T0zGrN7PKmxaCUSwUXlZdbgdkRzOCDUCVAm2OQfoPQIDMVSn/WxD686mbtKVsiipmLIlgFoxJPgwbsa22yQa11pSSKpHRBwDVXzpEhr3zMfhdyu8PyDip5qGNyzMB5XqFrs6JqhDGxYm1FFeCGyr92J5vsnwB5y++xKUErsDUEawZpTEVuRNPuiqlce+w4Ut0SecPpLcR64Q/sfx2PBP2EnbPn5ChgqHl0kB27osvhmt8IjQueVVf4Bv8B7K5SotRwSvAm3dns0yMJcS+Xpap5Kpx1yI+hQ2v5bL4yF9XRqsjHcoPnQvIoDkR3uWiwQHApguVd0SF89GQirtr7WvhwxCWpxGM54NdyX1a3pb20SuSCXN+w6pQ+eouKCXTKgtdILTQsBwGUFO+OORxSbF86fahornOOBBzLZD3fsBeghxw4v/2sG8hoo1V7iqomXZA5Etu7uNTflnI1SNHfT0NW13GAAvIPz+mUj/Fblj6UO3h7zY0NhjJDozW1NkYQ0u6MniGV3ciqUwlcnn6RTuKhEO8UougJOpW3wfSUIj+qFmGf0YM57CJ3BD6giQfcJUJkwhQ0+ThxIDmxKn/KdLRd8os1PmEJ5hZHPZ9aWjccLTncYCc4zgN7Mpk4yRVk+NaYvO0dhNoCx4NeGCeg1sGYb0bNw8xF0yPHS2u6i5LOzDQKrUr4WrpFUC9+OEhynKDKiQRlMH5AQprIYxi81qAhPq3aE1XUGOtUkrNiPieWh2ynlWxEDbBQSXNQHHvTQhLQDoeVS5JAsdMmCV6GTUijspBB5eNcVNyJcAOjR9v4+DUY78IB2ISf4+WUGtLDMZE+33MAxZUhTrOXkGYBA3Xq2SluaYRHU5ya3+cVN6McUyqNZzmXr74H3uJvaJAz0DAjpKoU/qgf1RSlYq9gayoG/yjudQQeYTusgnBMGCFnyieM1P+5UitXbEdiM2GhIGOUuySYg6bgXoo80yCkw4GXgUyaQ4AnOocGRVXq3cAUsRAuat4IGlH6ON6QW/DSrsxyYQ43VYU1KQ5QDcDSa7oywXXPdLf4PSr2mURHaaA7VzHV39BeQkWVVHt61LC3VbM/xIdrI4Q42+RdJpHyMEPt4fHBwIca4XOnZ6EDHgbNagv9hSLIGH+YMQgO3ACxY6wRN6OBHHAUQRaCSy3uWt8Wb5kyJkwslxNz2ZBZjDi7saJtaux+SQS4sXQcSULNhsJ4ugHRg2Yf7yTOF+0MGEPKjmtujZFQ8n4/QpG8UVjxiioTFDKM6QT1XKUegxDJod1w4JzF1SjGWAA9uk5q0SQ4SwUol6oprx14uzJgS0zJMsCM1xFSVV2TCiN3kqCeksT08ilPdxgljAP4Wg8qwfGBUeCmDpYX869WwXDgQ/00o0l/wZs9jZc1MatsFekA54XDRloVdS5aoY1gi3xEExF6HL1gotIo+zWn5Depe+yaBvzIBw1+ewh9CDqEpsZSzVvhPHRTP1ZMfke8Oweh/FEFhwDp1JE2AQFbAzG76OkkP65cQUUA3L+5eNRfh4OiqCd0ve5kngGTVumu/HhkQpcWW9SVULPuQfXQJr0u83vsehTbihLYx+JYltZuxj0cuacSWbZ2Xj3oGf4BBJIdwQzZ67urxeX1gl1ffvvTwqQtHcLZ9eI/1yH04vTbZjd6+OXq5c+XV3+wHxd/mHAvoQScts+7oXpvm/uxNqgU7dMFlbMQXuxQ8Lj2xtdbyo4C6W1JDJWpUzDh46hawoWDGgiXTOiIZ8ERDf0mgsVmcOTR8Iz0QTO9U9M2cjcNeISS6v3Q5lJRdgUZGl8tqAkptZ/ctfaNwvRbK4LQACdfHrkFh5brgYrD/YDMp4IzNZ7920Jvmn0y1mR9m9Acefyu0dfSgnlodhayr+D//B/n07zh52P2Axhdu+vgbFJRHVsiKZ7DvdIDZSCGhSlaAQcNudhy6J8bscUWVXG8aU10klRq/e3/dTMstLH4vcNW9JzI3+layhbDgx+Oi+RpuY878ce1jcwfzQQjgIMETFoPwuns+mATcDrdTmXWE4l0WkgoIK2+wXGHd2+xecunScZT0sau3MYl4nSazCgCvnz1ZnF1zV6+un5t6/bfLn/6dfHG/yaEf8G4XA91Isy6olY4kA1OmItiFegIryXcGmik55Nw/RsFf9T5HbV9E0NJpwY6EUYGRvbEQgq/T114X9Iskf2NzWczanJP40T94d8kyPZEALGEO9RR1Q4FI6RMz9SpsVGqDuREHYNAy3O6DxQdJ+LzQ6St0o/60xX1jLQ5VutqmBuYfdcKq+DBUPjRKBTqFIDN2hFfliHTP1lmRnBmmjByzV+uLv/18yV728l0D8FSpHvPjd1oJd+r9l7ovF2UFPvGtiOlmSZrzFd5tE4lCoia40wNS8Vhy/nDE1MDhiY2DuVavKnd+2e+GEO6CKiJg4KKVKJ02aVb/pOg1GNRe2LBibrtsmLaGnVBsfP9AR7I3IkutlzMj8qlJxUxjiNIVcHoPRfNOHxdteu+ibCiGPFhl9U3a/Go0XI4Qw4eH86cEH7KJmf/A1BLAwQUAAAACADnhVZdOx45/7wMAAAKKQAAEQAAAGIxX3N0dWR5L21vZGVsLnB5tRrZkts28n2+Aps8EEw49Eg+4ppdpiqHnZfEtZVk90WlYmFISIKHl0HSM2OX99u3GxcBkvKxlZ1KLBHsbjT67oZE3bVyIEXbPVwcZFuTkg2sqFjf854I/VLyrmIFv7gwz0Mri5OGVl8tXNN4i2nT2PXD2BSDaBtWEdaTlz7QOIiqT4sTL267VjSDRZlWLjR4d2I9f5xLmdZtySvHW9WyMr9hxe1N23AP9ElevO74DPjFW1HypuA/AcBv+Obi4kKdlfzCBl6+aApYlD9WbXFLmyYFkLHi8fUFgb+SH0iei0YMeU57Xh0ScoOA5jX+9WPHJY1TBxZPrwAhVfAkU8JOS847/EI1lRCSDQNvUGT5ERgDFODmn0yymg+wgxbdOy7bntI4nuEeDp+J5Y51aOUdk6U51X1C3rJKlHnNev9090Dwnny7yuA33gHxXUNnz3nTyprexwFtR1ryYZTNRN0dIaALqzR8tFThLBd4kKNs73KutUitUZgjiAOpeONWUwMWk79l5Nl0SslEz8m/WTXyF1K2kkY/8244EX7fsaaHAwOzb0YhwTuGEyetFEeBht2Le2JIarvoI32+ij1w2SekZl0nmiMIcbdP4H/1EiRPhDEkIhqgMNZcwsmXfE4saoop0ONNOTcfs419+z5q+B2YY8nvo2slAI0dk0uySUhk+XcgwE2Eoi9zNE5YeMmqnn+YNgA5CmSVbhPy1ONqydnSpzSvcYDzlzP8pxwtv3MhapfQbv2r6AdLOwQuDqglE/Sov5yQxppXrlGz5xrZWLA5jDFHZTNlPsLJZI7Wa2kBHXXIhsvs8dV3WyPGBjy1VwaiHgcmj3xQCzQy20aJspfYWc4nrGa/u9xeh+S+VfRK/kX0DLhPDzEOrBbVA9BAZXh2bPaarMOGPu26Bg4sqRdNP7AG5YxryTIQxwQCONeIjhwI0xGD7269rcqE3IkSHDZDoPRuk95xcTwNaX9iHfet2KmA/CNDxNCUl3Hg5ctXXhSoxx6SVVNIDrmEaDpq42gy75K/FQUouxweOr5gyL6drSLsdE4hYRtltb+KhjNJ1R6e/YDQBesz5aWJ2TELNs7UvxNXw0nI8i+m2YN5NCFRj5qh/6VE7wSo0dQSbX6UrKSzeKPEYyS3uwYd7lMMAjkNZBrGG3X8sziPV3H08ayKMIX6uX0BsbtOyILydkFZM2kMYGs+H6Od4LESQzPRHDssFSRstDxE77ULfkjfKx/8gO5w/R42/3D5/Xungw9REKYUDVf7/NwW4O/N8Nl1j9Jntn36LCEnzso+e5JoD8i+e/YclCnbrh2HLN18QW2kCCX6uyEAkjCrZuFcjYRVQDJVDvhoTBFj9CusERTLUH2sLIZU39y+XfONx1CKrIEDV9246qFr0GNn2JzqsxBJSdGwufIipFa2d01AxHe38+Wdp5SbBDPs0bGLFd4sUL5JCERllMmOpVD14Evq4fnK0zQePfIW43SQEDG7tucUTH0beo11BMgkDNOG1QBdUTCWedAojM0tfZyQy028n6K+0sDLtC9YBXVA2Q55J8GGiyF3NKg5iDOmvMsCY4OMoJ6BX7BQKNRU2rlKJ45tAfzS4lDPBCh8zM+6Ji+tnDi09CTcetryHWwZ2DVIYa1qnrGElkFBHqIaqbM3+i6ObUE9dvj0CSZsfPhx88cwlg900T19pDNyhQ6EsIM4ejZnDc2vq1JoCFQ7djZaTASjm0207l09cpnrDVWXhV9CENzFvdpFaKJq52gfwik7g7ZyKE4euLc6RyixScinGn+9E0Ej08RSKI9cUZcrbDjVs5hkGXmuTW832wJNYNrgbG253ERDAgzsoNJvvNjAGr+SDzREkeQHLlHZ0Szjgqti9lAdQNS1bQVElRzxE0xfoLPhdy0sYGTg94Na0F/RNV/zQoOtRQP958CVU/v46Ab4XJq0ldfiPpoVBtosK/B+aSwSmZ6nbDjwxMxaOIYUt4LjNrbHDVC3Cs2vdj5GQreKi75kF+ZkXZ7nKHSIMEdOt/E+XirtKMe8baqHSOs3JPkxLgBRs/DL7/+iivvN9jkeoYRG14xuMuyr0LPB+nNVoaiVFTZUFASGoYfw2XmFA5rzPKxl7U/I3wV3jfcHNOb4zCrqqQNP8+yJzql/suZEg/T6DEqXTbyyiQp94iBs0zgjHrCoCP5sAnC6eRrsoBjYBuMWziBwcx1IjHkKzJ9qNOJZcm2jv5tvuTe8buWDwQCo2jR81KMT5K06HdrbnNc3vCwhgPQIOEF8TX7H0pR0TL0lBWuaFtubA7Q/EATMPqYJJNbucQQi2/F4AoSxxyEIMHvptJJOlW4rp96wts1k6LKL/AtBC2AxO2DpL1Dh01QQAJYejwedQKC5r29KRliCdc5NdZspFq7xK9WLWlC7q72aelmhjj3PJYf9wI6HNRdGc17fXe1APVpuBwf+FlvK3moWtMJNZndysYk+PVQtA8edNlchsMFiR9EFIwWjoIv32H9c7bEkQt6nuZEqQXqH3Y81hTpqjo1lFKhQ5ljEUIt0CW7i6pnLDXqNzSZBU90Ojg5rHpB+yqpq3rgtu+sXDFKtyhnQh/Kyh4LQnlbJyGurfbf3hNgXreRUizdOe3BX/o4rBowcUxw6Qn14EMDQfwx1KCV1iwnGDljtYajZfSCWmjPcyVCGCsqTMvyHYW0fK2FuYvJokqJa2FkIb3rXGwUCtZ3em+mojtMu9EvXHsODGRPECZk0sfUqg7GumRTKoDStfoBgQSlynXhMO6GtsZ0oruDjalFd0mmHb3xpQ9J10rpyxwzoXp3tQZSeIVC5QGJP7JmJNSYFSiCAoHT0k5rZ4qPD/6R5/YSIj1zCrkVfqzLOj4E/jsUtDshQgxCD7Nb6tuIk4AXD0QU4AUialyBvYlMR8KcGvxzOyQk7HiWHkjuIgRoCLLaVAy/ppHFzJpD+LX/ITNAS1955d2If+4rpx0qPBZWwQS8emSDmtodDz4epbLhSVkQVK7b694ramRThNSR/jvlFYew0uWtD9dsF/j5A1/aKuQdMJDyMnjaajk/tEQZYsHtnz+bCAgk4YOsMsZ0caVCIhsfZaCmka9OlRzmvxK1Jmz6pG6hsQ1xk+XWiuZ4GpI6lZT7o+Ru3FetzcD9Q/SSEL2TdiGUHLFxrWbyJ9yoAvlnAqWMuILFaC0C/JmbOqlKtTe6mImWVhKL+gfhXcw30KRz+aYbqISU/mAHVjKaHwKA4AMp4T4KxBzzDlD5QaIBl3LVjVeKcHRbho2hr6KNnXqMkr5Fczgzrp7WK53/Sl/ErUA2O4Myeu9eTUZ9mMdYguPziACHMrGjeD3ZK8yc97J2rfZZlV6lChqZIEAAgH8CXWI1E1vu57CP9HJ7pZF3ZcoiUQ2F6o1g2Di0EwoFqdnM9uVXfU30c3rCbipemcloK2jRyCTQ0mR9DzDo96bi2LH98pjWe1xJSd4ygc7RZa9l9UrMExdlsj0WBgX0u9SR/SiaZT+WFX3JMqTEMqefSNXA/y9FBIk3IKcWMDl/DLI1/LgllK9na75JXErbdAZMP70pR66YuzKa6EnBDPWppxoFrzPppeoqtQhYNc2hgQXew1h4vTVbX2cZQgvLaWn7Q20LuxloCoa//f3bthODvvbBmYNbnd4mkjdm7BlHXBv0Cbhptnj7bBn1VTh2utj6zzycsw161QwWQq1YtBwuG0kLN76CkwAcjm6+++uqHqiJtN4haQLLFodjY9YTfCyiBVVkF4HL4O3xHplVgkG0F1V5TPRDb9RG8Y1NZAQgqwmqv1N7Dq5uhnE52q1+7+eIMzutZwrY0xDKjuaAe/ZzbytAs9A3lWU7RWFFg5PuMbCbM6bZ6d6OZPM+grZN93tb4miyugKSMsWI7bQ8ZYqOHNE+Cxa1exCrCXiive615u7tUxK/34PkVDp1mPNtbZAf3RdJa0a0doKLPfBGi32qfRzTuslMTTzv6hNg/qQMXyryzP7LpqZqHdSHJvfGZm1FUZtJNb6Ayqubz8WnKbQaQ3IzcQV3BD50cOk7DPasHODutX4zgFZQlmI5difXQAC1xjkE9W5mJ25s+w0ww4870+YO19XGuN7c2SN5K4n5Bo+VixveZ5XMX+cvA0uoWesTNy5DGWwb68K8ZIHVOmsowunUpVoaVGbB6evU1aqqDMxHvKvwBiF607BvFuxCY6xBocQPlW0Goi/jlz4Q+w/Qmb6Lud0kYHgAgVZG2x3xHIyeTKFalf2R+URXpSKLvhydBIwi+UHSg/tdUaKR+ohUlsblOAK5jey/dhf5TimKgitM+m45ZSWd0lqO8kmh2eOkjwCTOQcShGfjkgYuAsjLns1TdW4iP/wVQSwMEFAAAAAgA54VWXeTyekyaBgAAIBEAAA8AAABiMV9zdHVkeS9ydW4ucHmVWF9v2zYQf/enELAHSa2jNsMehm56cBunzRongeN0KAyDoKWzzUYSNZJK4n363ZGSLFt22xlILJF3x/v7u6NFXkplPK7WJVcaBsK9f9OyaJ6lbp50tSyVTEDvVrbtoxF5y24UT2DJk8fBSsncK7nZZGLp1bt3+DqodzZcw28s+VZClMg8l0VDxI3MRcJIkaFXyOfBYJDCynuW6hFUQBLDdwMPP1ZOhCeKlte+MA3rHApjiRJZrMTai61hUSZ5qgNSwwmKFPCUGXgxQRhaeiWlQWpL4njnvqoKRuv+wtEYtXUaWAbQVUYse2fXvGFL1rEqsGe88fxvcskce0Tr/rAW5rjgJYHSeGP7JdA/XHuglFTdsxOpUjw7FYkJ7CYz2xJi+ufew4ixgufA2NDLMX58DbE2qt4c7gIWt0/RSqqcG4YKBEix4iKDlFUmiTEajZ9O2UTUlYKdQaThjqVUAp3j/307/TyessvR1fX4wvO91y48aZWXOrDGkM5xEwB68RdD79WrWh6plVV6E89UBTvpGAHMZJcvGa+KpImhHnqyMmVlbBzr/GlyRqpk4xZWmG/1QpRUKY+EZvwJLeLLDILQk6q7mcKTSIAlskKTQu9PL4OiOS98t6+UN0UirJMxuT3wp/BPBdpA6gktM04PH+8eNFYjWBXaQ/1+VnYNaTej/DEVVB0Kk09btwwxg4Q2TD52vIQpp1HQfOEqCA0SRQovw6ZOsJSgqHJQqNMRW1YyS0H9qD52lD+rVp0akkTXZ2AuyecClN6I0mVTS4lxssSRlaSDjoL0sWz7Fe/Ie8XefAiAeJE2dvlvCOz8EHWgUrHy5n4pUjSQ9EryNBMF+HsyUKlaTKuWRzJJws4kKnnnNmdTGEGRyJSSC13fCLCKLrcGeradSqjZRiA6vPDEUIg9npGIrYcg63EvE0/gvOJ33I2W7nn7ULWjZU5cTbLsZEHxhKKkRmOehEKvJ7LcBnv7UVWmlFMfHi5G7MvV/dX76zG7GH+5+jC+t4Bk8xDr+u7r7NPtzcPN+4fLy/F0fBH754gjt5M7dvMwYbNP0/Ho4j72f8XFyefrw8W9M+e+E3Y3mn3yF6ghnWMjzNgKQY2xMKpTc36OoX1NJpCFGkp8Ia3XgGjVkTL0/M4ZmaTO0gmva1KYdGuMrCwRD3yOPDbGoljHfmVWZ793o+C6KunWttjoznLOscViKkFSGUICPPmsovPPcvq/PGfaVOk2wuKzq2cYP3wgE21rWwx7iXPko02KeBKjwsSKVqi4o8j97OL2YUb6P8X4RyRcGVbAM/Y5rbEpHdTwXqpQzQ1da8LaiRuh+DzE9r/VIuEZW5dVXEOQFd7tNE2ixa1R4alOcj8bTWf/p5UcU+BAy1NdhjA04iXGKHUH1DwN75Dywrm01t99YQ9arTSY+G1tBknArCCoQpHzdw6UnzeYms1eF3oRqw3khBP1JnLso8M+PBP13Hdn+4vjMG0PFIgE303iw/wdukFEx76CMsOxAbESkQbDBDzvA5ZbjzTAY62Uc8ShIrakEFi1K1XiIRizS0GfdF/UjsdAlh2Qk/NIDDnPiusr+Yt3KdUzx3EKQbTiGZXmWlFtYp5Jr6yWmdAbSN9YNHWwQYHWUU+US81D97vMw/z0bZaSGpFClUUZnEg0+lBzQNOcsDrB/EVUyp6NtgEhMTYCGiFuZAF9K50cCu0iSjKp4YhfDahcFOiBTncIesNqe14ce289yLAl7Q9/fcHtqNzpzM1he82ZZLcbbTe1Z+wm3difCMQgrJ9WYTcd+nbEMIy0i+lfX5Mjkyv20PF0cnUzumYUntPRq63bH9b74ff/un3PGpGncen4OYcWnMJyhEwDcR0W26vsCjUEN7aTnjhnlns0Eu8TufgXFHNNWfvhKaxrbXKIg1HK5RPY3N4R0Tx2CFc2kXA+iXQGUAbnbx25rnDCUduDEbSGNULBU7Pmz4DZqjfPdBO2Jat1aCC8k4yrXhaujqXfdwPnQtK632l37KZ0P/44Gd/M2P3DZDKafm1uTPYE8kRc64mBEYWFns4dLGwuLLzYBqob+tBWP3ozQOXyMgOXDiWvNOljHU5Tf+OGH9xUbhE0kSOXyo2QqbtCOPP+wFwHDYoAscFHNICvC6mNSLS3hBUxYpfgohk+FZhKtcfjbQ2taO6oBCY+YzndpJnvNLM/T1AeND9VRCO1ruiSfWd3agRzZBFPU8br/aCdihTet4SCtJPd7e8Jls1+ESNGOkKecPAfUEsDBBQAAAAIAOeFVl3EuVtfuhYAABZMAAARAAAAYjFfc3R1ZHkvdHJhaW4ucHm1PGuP3DaS3/0rlAQHSU5bmbHjIOuNjPP5kRiXjQdebw63cwOBLbG7lVFLWlGacXtx//3qQVKUmuqZxLcN2Jb4KLKK9WbJ5b5tuj7Im/bwoOTnbW6eflNNbZ73ot+Z507URbM3b2pYt12TS6Vsy8E+9uVemudPZbspK/lg0zX7oAV4VbkOdN8FgjcD62HfHgKhgrq1cJou3/FMekzq2kzdDHXel00tKpzyhgep60qKrk72su/KXJmx4kZ2YiuztpN5qWBSpvKmk6sgr4RS5abMBYLKOonDVwHglYkh51EP9M53Qslvs/y3ViZ5s983dieib/ZlniHZVkFRbqUCEDvRFbcC16ib21WgYAvwt5RFJqoKnnYC/upFP6hj8PIjwTXgh77JBcIEeq/FuqzKvpQKpjMOSlYy77N+10m1a6piFdyIqixELzMg/Sq47cqeUC9KIpheMIERwqwBvS1sNsM23b1vClmZ/vVQVkVGTaugaeF4y0+yy7ZdM7S4Ewnrd6IEuvZA51XwI6xevK5zmND9R9Xk1w8ePCjkJlDN0OWwDBEpip89CODXNU0fpMQLUZYhr2RZnOB+6l5dnl/xINkPXR38s006WcFp3cisbyKcGidCZW2jyo8AEAkbtXGwaTrgtfwadhOUdRCF63PY21AcwlUQEq2fZF1nX5jwYUxLmR/BwNkKSCCLiFYLvjFw46TbVs06Ch+G8XRiuQnapFSEShQHIDZBmGXtIRf5DnALgSV6hNsikr2iAW2ihs2m/Mi7TVraaNJ/7MP4fzXxctECDWTW1VtLOqYKnmzUHvpdU6cspclW9shesP6KBSutgXS2L7OdJFcpSxd2AHTbOcEKf/lQCD0WH6cTkLMBXcDeGQFkEDeirMSaaCErJYPLq1ijBCwLYxklgmHw4o0qgwT9fRkyiuEVk3tESFmE9EDC2Izj3agJbnocdYVXSd4OkT7EU9t/ZgniDFFHNLhUDJA4SBEH8XI4Pryy2GstFQF0oMsqqJpt2cO/VpTT5Kk9aDVUKCUk9MszGAdUFAoGTxRGxIPNCFnAgIhHPk8dCCBO/aGVUVn3sbM27B6V5W2pZHgFU72q024M4cP26C29PFsF51egOIa+HfoMuTX90A2gJ0CJNPB+Q0o5PZuup7UwrbapGtFHE73srAU4xNO5bTebumACFoCQVDEsOKt/17anAYUniiimw9MKNdJKkfgY/mmB2GAF5E2ZyxWpNn2CNC6RoJsjXuVaHhTp6gGVOe8DtnsJhOI/NAo4B5UPdKBNBa0MuqOpy1wDQQ4rcZ1bZDMJfA9oEn/DdhJoVhFtKXZ497bsd9aqRLxVpxt/zCqwKO0a6H4Lx78b6msVXtFqQGHQuGROsEUDSZjW8eXZlQUH8oTqjklYqk1ZgzkyvJiQypiuDXYEdMQbhFTW24sG+PB11zVdFP7S1DwdcN1IMA+5q7KRnoloW1kXvOGyMBoAf0xoM4DXZzFNSFtEjhLnw5gAoyYXHuAVlcHXwXkc/FtwfnYWpGlwNsWE7TuZDVDmePQD4QSqfXzJ6HxSzTi6HdwEOBypUloBlXQvKt1UyVofb1moS5p3NWVc5ixQj7CB/DpizGPDY6sgmrNS8MjwWQz2bS8+RrAmroOQYqOumhZ5i1wuFKNNVW53c/73MP6XX375ogXxugEurgR4bcB8eS7bXsD5BWAR82v1Z+AR8B/rfif7Mh89jGBo0ZFJAIZfiJDDUWgsv0chOSJA4mvWv9esfw29dPfV5bMnV1rCdujhpJpDe1mDuY8uL4EC366Cp6vgu1Xw+MryeKpZ/YHmqbKwU5taqqwqr0H6duRGFqhHtblcN02lVa8oChLoN2D+CzM2Ah35JHZHZAY6vya1jLSCJAGe6aSR874K3lw8eRzsQEDAMQMPAwioCAhwG7j8HWjAHraqAnL0qub2kVWKwDpdDW4f+HU1Dk8sVHBX10YfJBspyBNZiz7fGQxou8BlviGMwmqCWexXEaAR8qpRMsIVV+hcV+lj+QjOojOPXoXxfqiRrbWquGAio2c11JripMACvTEFTpMS2066KgQ2wrsvgD132R4UABBhutpe7pvuYEmxBglbw8knfXOdyf1aEpkVE+XYn1yjM4wcOZst2VeeLuXQBk6nViQxEYHwuNjx8eTJfmlexK/mtO6Fmd5bVjfdXs+fThR5P0AYtjBxyh+uaThx9gRxFZjdEhecy0ffaS7ARw++Pk74O7gYj7ZArADCKlETl+c7UW+BI4DvSR3Ao0YS5GMPPgqpOc0YXwUvUUcFAtQrYAnasdwPe5CZBv37AJUySyTiDSFfsSKMBBjs+gDcR/EKrAAMIkETyX8MaLgSo76QbKBwUZGtUHOnldivC1gMYpkB6C3aCDQx4D1a4FjrCSY3YRA5mmHZtLMD9ofN+mgfG6VIgeUdPGUQp3UNWFAGv5ppUtd8HqlR63Pz3ATCt1YGX4BnCvr3cRyAyHi9B6UctHzH/rZm9WncVDhaXkKfKoJIzIlp1FCTkjVpE3wcA0AmNKpScNllpzjKwVD0H0MJuoR0sA7j6LFUtG1wWOSVwRAbeAWNFPo+aBaiGXZb7Rax/driBmiepYaoDzxxm4g1bCZBRomD58HZ0ZTTRPpLCbQBo0BwjW/VC3VNs0uMv0N7QrjqMXpzIlnhn1MLJ8wodsfu3nTNJ5AYCwhkCUQYjJcsqgMIYy5L9CnmW+V9UGRBppGSEw1or1pS1MGjMLlBJDJhyYpYwu9l3OGEn7bH5KL3co80Qimfqq17OON2SxGCcSWW330i6w1UD+B7dbDvT9LgJfOmJpa/wyVEXxD3btxM8CKQGpx3YM8iK0HiulLUJrZjdsqQnTJzRqaLTjYbWQTCR8UnyyM8lowPFEPssgBIZX/IIKRDyS5SEgaP5QZvxEYKmUY2Ay+WPOlUN/gWa6WATbdDtj6A8+RmPEBXZ2yVMNSHU8PMUGzVmkn7+ZInFJdTeotMgUpR7y/reGKcznBNvLKzxzCAOvhMnCRkRJk08sDh6MIfL/6WXbx//ebntz/+9CHBAeAc8xGaXACEWNF0XBBC4IFjkwKCI4hheDwcXTWonSNHNlbGfh0ocP5yITLIUdFsOULgFKjNYYJnqHrZrjgoaDHyy3ZC7WjY1ojFpqnQUKeBTseFvJwK4TkKldzugT0y3L/qu4iXuzTtoQmVGEqyvy7KLtK5Rs2d8mOp+qy5drCcxupbyrJmlGDxJQHCHEzPmmOlcBYTcUbKJFuyMa8zy+JGW5PKxdhtOitIjyBAXGNawivttrzBoSSTuBHYhsq7cg2e7wgI6ErpocOfQZcERSNZsee7Brww0PgNBGTdOD6xpBs3yjkLm4zATuJZ9LAi11UvIaLIJQ6HyFRwPzIxIhpjEP0thtA431Gdk5VMQH5EKwSoRNeJA0O71GtdGQLahnhCqGPR0cz1zeQMjdDQUfPilECiwD2EOOI3ZNWAHLFgcvbHepxTfun8/Fb6eFrKbM/wTmfvsc1/4rXCpwx9VuhToIm8+88mecCkbj8BMhAMpw7dKNI3CcV0mpVK5/Q1JvSG5QIezAR4ZMZFn7fOD8syAiO9sgEtmbm+SW2K1MJ11polPZ2Jl6HoM9BEAiibOed9B8Dkqbbn8xsTh6y4ythjOcNDitn2/FyG4Ppu6HcG0iXRqizSa9YzKermg7YG0HZA4fpUtpG7JB2K5madQWq6U1cquLR78UFqVKeeMu7AK4hZ6tne+CXdUEeX6qAS+VHmEC6tK1TioG7t6sBL4aNHhFrIXYtIe02l8wM4I8W9wOYHEoPgm9sS9u60au/lx95R6w6YZrMp81JUmcnMASy6fUmYG3BmxKRIVF8AVDAw4f/UaGfGZtl1Kw4lwftIw6HfPPp+9KD1ODaZGG6e9n/f6S05R9rJ38hO4qEHDoOG99EITKk7NYFmrFEZODx9rA8M9/nsZIYEB1N6py7AUV5lcBohAv4Hddvdkn4s5VPsjtSPQiGx86Pwbxc/v3vxKrtgOUNGMc5IDW4vaCNgIO2n4DUlvCUg2PoowbMHr0ZQPgZBJxgoZHxHiPeD4JqHTvShL9mTv5ftG7x1tNOBurewddv/9iJ7BT7eiw+vX8V4cS7AswWTM3KibmC2P0mOo6bZzoHR20rkMsL9cx9newEl94qa+leTHC3yw5Uv2GAAqwApmE7JuUIFtceItqlT1BG07Mq1C6nz7FM6dBCpvsOe+KTpzEO1r46rqnbi8dPv0rnz6llnvEHn/LZK9QUhAkdMkN0rifBl2+Q7209vhKqE+IkcDgy9OCIwMLCLWmCgZ/ETERGRGGiUaoll+U1dIfbHNb87XMLgUO0Av6HP07q5xbZdWUBkl1VSAMutG4ik+I4uxTzDKSOq9aontHHHm3ABEQF8OGo4EQ+hqLx//eLVf39WLKTLIjj60NGIp+7BsDIY1gzbLffD832DFFNd4sQ80jq5rFvtXQW/RnwjbQ3UmGo7efntM1YQPQY6uVMYRWDv3ethT26YKFT0mDvXQ11Uco4+t5o9a/MghqLs2XjYIpXIhJE8w8n84B0aZggIWaduJeKRJgCdZFAbk3hx9k0UAM6C7RN7ay5GIVM6fVLu9+z7wEr/vH4W3Bgn7QadNF4nwRyNToFdm8IPG6aiFkXu3Us+dXw1HICau+mu9YvfRTIEQ38aqGIngpEAXkDjBI4KNpiweNcMneIGUpQZxFPZUJcfscwEIW5aq2pp/6lFEjQpLJHSaVzScqSIMAjK0ODqDttAvVTvA6ppWvczx4ZZOzUHdxlyg9aoCIgyJqgxnUG2zY5bFnp9GFraXRbwqoiylmADJnEfaKyt7Eg3pJt25DK7Ic2oTAftD40XmfaSEFuSF4XY/1c0L6QybpHe3iq4lXjFCpFMLg5pcnauxQJCOwei2LfJj50o/krNE5Emi2DOk8zYGWwcbQg+mLwhPo9GA9/WqCA35+mj8wReREFOMfYcc6G6LiEoL6whg9nWLGn7kp4lc2s6anTWxrykk9Ph5qRF5jVta91io5L9kf5wZemKZE4f/FaCRnd7dSkSbsR1z/eOJ5SLuiAnBffFy012Zge24lA1YrwIxpfIzl4Fe9FmZPuQ2GHeDqE5WpU1dXVI3wB1nZwt3i0wyMvQYTvwGL8AB7P13X7+ik6pVsXvea+aib9hCnzDIhiAg7THK1nn1pOVIC6na5h0WlfvgLpJmNFvcsOnCYcvA7BD3PIJZuLlSdw/maGZeRyCDeF4nWguBzKqlqB7NQgxc1lW0bRmwtQAYHrUcI7I82E/VJy3gZ6HtgddGPbCwivvESnw+IqBdosHNArbzKnjIg58U+l0r94Lbc+R2pU0MTyH2dVbl0bw6uwaLz1014iJLt7dDX1ZjcSm1wQLhB9HI/OvrVCurU+PcvRstr4VK1snAo2/u37ECqZ7B++WDMIfswcYNiH3rBznq+DC7gQOFvFRwftffgQPoQ/2A+gezntiVhIiO7osll1JNlpZjabrErF4YqyMOOl2oQPR3Uh9yfunM1SFFvfLcNkXBwweemz+nJONiQeO/TqY9bmZSF8/ZuB87Tr6Qhk4T76H/iew7WVjuR4K0K4TW8m+daaRt3ZAv6MxcLyRdJai1z6KJ3Y5Ebnck6bxve7z8qFvNhs8s7LmizH8K4qdCH62VyDVk+/OzlZO/8y9ugoeGWawfK3X+UFvgxe561qbUgA5XqYFehGaTa6n5dMWIpNgL4UasM5T15Pr5c0FuAAbz2XpQLybshkUV297qLIaBdpRYv8aDaujFRDnWrRq1/QTkjj1zqQ4UjZejgWJnQL1dLRO0xFsXVJthCZ9XkfbKt70XtrdeYbFQEulk9rthTUQXooaF6Iz2eKDrommcPm08xmPdANuV6WakA2POTKe1kjX0Wp4XWabx5jINn7BQekVhofE9FCDncxZ1sKLtvVCTY23fkconE7QHhf2Y60DcC2cKH9S4eQPHGx0AP/yp9cv//Pi3dtfPmR/ffHr61fzOJ7Q8SeSPBid2Es8ywXYozBZLvcsjKFil5SkUA+jOyVcLUbjNVl+5h+QFNimbKwDv/NmcgwJjy0t/qzOGG9K9H2OXsHckWhUzHgU4rxrIGgATffcbF7HEfP9M/4cMUQm1vBBmgYf84ycS6BpQRrx+5q4/Q5V8XlCNyuD8wiRvS2c3iHyzSFF6fracDwYk6s0BFm+HfGxqU5C4mcw5KI5e5w6avYg4JgMkTEnnQbnD2b9XibFy7TJ2nPx+/XFz29fvfjw9t0vnyd4XlVyRCWvDPbdwSl732G53hRO8IPXCE2phF+EOE7A81Qb7uMqnXUnxfWktem4VmH8VsVcS6IlmHqMfBPg7C5OQJfvdTXikoWd8uAEyVGVAp44nXbDHyS5GP2wiJBxCM6CI/96pNytgE22JgMAXRRZo+/0ZBWcH2N17Ndt9EJfpDNfxF/SityiqGZi8tnZqOuwWMQ70yOiumCcQdiyEYZBdzL6jnI5rqPtpPT3HTeZQcDSjuk0p/opxSKgFj9DkJX+auj+ZYdx7Ef1yKmjfz0+tanKIOa4PGKdZ8fM9PWCT3d1BP24QtX9jd7ZXdV67g/L9DKgGLJlctTbl/n1kl/v/qgqry4kfXCnieDnNq7R9Xxg4PmoABUfQY09bI6/e5b7jbh+1nc4XrgLxbv2U7LfW7w7/y198DMp2T0i8j0++kEQ4TJOxBNfm++9qL63kL3Id9HCRo33j//QeF3oqLkhjuf1wZ7JQ00P2ZjLPR46lhX/f1YUuz9NoHSxpJiBTOqK27FI2KeQcTGe7D80GzrJ9hTuLqnY31sYo1XqNJs890dme1yc9Dx4fPZHeO29bCXV6u/JMxs/i2luZAd8dbvAfqAOAdRwrF3NR+sJZtNUkldlSydL31JkkS5Axiy7xB1k5SazhdgL6m8jci7uQQOLM6OpXaYv0r7BL9IwxI4SNMDJn+AxeYrv+MpRe6MiemhLaEdgR3AmQS0CdV89QkW153iTgaw1Knficn3F4T8V6gM905FXqd+QCFiQQ60PNd5LcngXH97Fg3/MFM0o5mNVj/VMJ1pmEahzjX/fOfObF57pKe1GK7nghjkI4XeNlEfFDwSOsqvn/rNsu2aLlTrm2ukz/fwT6YE7fS2izHix5auPIMuRWvMxNQCrSUHIPZcbU47MbOkC+Ve/u2piWXOOH5kaDgD35OFDcxQnXBEbqJqJ5O5WYZw0LVAiFCFVCXlCA/Pb6EohJ76z6+oSuSWPf4nVPBSz2UoMv86/P1tQ7zb35fcSLJB7OIieLT71fuVrfmOy5w9sbVzNRKaA52NyADxBOoagWqJaMGL0IfTV8hEdh6Xjco5iep46QaI3FnT3d09tB+Q6GnWCUstU0lmF1xfvXv6UvXz3l4ufX394ff/Mgl+7eBJJnrqef+khTQ/njmss/PmuBX5pzEeY80+nx1Iy/sbalsihoi6G3FbrLBzM8YGYLyZTDzF8yRSU6Ohz6TYun+8aJevpXTun+n7vNfsknYj/owVBtvfKR3kt/z25maWvyUfweNnKnc71ha5l/OwvZGx+j4p9alGFlKYyH7NiAgaoN6jJ8fLyxgni1Gdo+GMZAAhPJVqFa+tbL3Oht/zR2JK1nOdUdTHecRUlImhvGI5HncrGn7zVMCgs1wwSa7hGlTstX/i9F6os1Ynr1Mxwyk2Rn8bEtnsmbs2gJhEYfSdxyXkG/jB6yl7z/5DJXGzZQdsc/PwKOccV4NHRkPu2P2T0/xLBgP8DUEsBAhQAFAAAAAgA54VWXYpFIXs+AAAAPAAAABQAAAAAAAAAAAAAAIABAAAAAGIxX3N0dWR5L19faW5pdF9fLnB5UEsBAhQAFAAAAAgA54VWXSsCS0SPDAAAsSQAABAAAAAAAAAAAAAAAIABcAAAAGIxX3N0dWR5L2RhdGEucHlQSwECFAAUAAAACADnhVZdOx45/7wMAAAKKQAAEQAAAAAAAAAAAAAAgAEtDQAAYjFfc3R1ZHkvbW9kZWwucHlQSwECFAAUAAAACADnhVZd5PJ6TJoGAAAgEQAADwAAAAAAAAAAAAAAgAEYGgAAYjFfc3R1ZHkvcnVuLnB5UEsBAhQAFAAAAAgA54VWXcS5W1+6FgAAFkwAABEAAAAAAAAAAAAAAIAB3yAAAGIxX3N0dWR5L3RyYWluLnB5UEsFBgAAAAAFAAUAOwEAAMg3AAAAAA==')
assert hashlib.sha256(raw).hexdigest()=='42aaf49cad4546f0712daeb00c75d021bafeb21e703da3785f6a42a177430ab1'
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name,wanted in EXPECTED_STUDY.items():
        data=archive.read(name);assert hashlib.sha256(data).hexdigest()==wanted
        dest=CODE/name;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
sys.path.insert(0,str(CODE))
from phase4_cjpe.common import atomic_json,sha
PERSIST=Path('/kaggle/working/CS781_B1_Hierarchy_Depth');PERSIST.mkdir(parents=True,exist_ok=True)
WORK=Path('/kaggle/temp/cs781_b1_study');WORK.mkdir(parents=True,exist_ok=True)
(PERSIST/'STUDY_SOURCE.zip').write_bytes(raw)
atomic_json(PERSIST/'SOURCE_MANIFEST.json',dict(vendor=EXPECTED_VENDOR,study=EXPECTED_STUDY))
FIXED=CODE/'FIXED_HOLDOUT_IDS.json';atomic_json(FIXED,['1951_70', '1952_23', '1952_50', '1953_36', '1954_175', '1954_39', '1955_23', '1955_59', '1955_88', '1958_82', '1959_124', '1959_145', '1959_21', '1960_31', '1961_113', '1961_187', '1961_23', '1961_46', '1961_98', '1962_203', '1962_310', '1962_55', '1962_89', '1963_111', '1963_151', '1963_164', '1963_259', '1963_280', '1964_117', '1964_119', '1964_123', '1964_191', '1964_324', '1964_61', '1964_81', '1965_127', '1965_176', '1965_322', '1965_47', '1967_12', '1967_159', '1967_171', '1967_226', '1967_309', '1967_72', '1968_230', '1968_248', '1968_4', '1969_136', '1969_311', '1969_339', '1970_176', '1970_320', '1971_286', '1971_522', '1971_536', '1971_551', '1971_586', '1972_202', '1972_247', '1972_412', '1972_557', '1972_60', '1972_81', '1973_151', '1973_267', '1973_80', '1974_420', '1974_77', '1975_152', '1975_28', '1975_413', '1975_499', '1975_64', '1976_197', '1976_502', '1976_72', '1977_113', '1977_132', '1977_184', '1977_51', '1977_77', '1978_110', '1978_271', '1978_277', '1978_386', '1979_227', '1979_422', '1980_275', '1980_379', '1980_38', '1980_458', '1981_19', '1981_28', '1981_319', '1981_343', '1981_37', '1982_116', '1983_114', '1983_124', '1983_289', '1983_302', '1983_318', '1983_344', '1984_10', '1984_102', '1984_141', '1984_153', '1984_186', '1985_370', '1985_42', '1985_61', '1986_100', '1986_144', '1986_18', '1986_206', '1986_241', '1986_341', '1986_6', '1987_176', '1987_180', '1987_206', '1987_221', '1987_551', '1987_577', '1987_74', '1988_109', '1988_16', '1988_230', '1988_333', '1988_382', '1988_418', '1988_70', '1989_140', '1989_167', '1989_46', '1990_12', '1990_134', '1990_215', '1990_254', '1990_380', '1990_481', '1990_590', '1990_630', '1990_642', '1990_66', '1990_69', '1990_711', '1991_138', '1991_16', '1991_167', '1991_201', '1991_279', '1991_311', '1991_376', '1991_462', '1991_71', '1992_225', '1992_495', '1992_652', '1992_67', '1992_715', '1992_723', '1993_139', '1993_200', '1993_290', '1993_432', '1993_494', '1993_544', '1993_608', '1993_653', '1993_696', '1993_717', '1993_87', '1993_93', '1994_226', '1994_295', '1994_501', '1994_692', '1994_909', '1994_980', '1995_1136', '1995_1264', '1995_194', '1995_266', '1995_468', '1995_488', '1995_526', '1995_547', '1995_549', '1995_761', '1995_808', '1995_972', '1996_1052', '1996_1080', '1996_1345', '1996_149', '1996_1529', '1996_156', '1996_1813', '1996_2058', '1996_364', '1996_406', '1996_412', '1996_471', '1996_555', '1996_556', '1996_559', '1996_661', '1996_69', '1996_889', '1996_93', '1997_1176', '1997_1183', '1997_1247', '1997_1310', '1997_1500', '1997_1537', '1997_1588', '1997_1589', '1997_1605', '1997_1626', '1997_216', '1997_275', '1997_358', '1997_363', '1997_418', '1997_446', '1997_501', '1997_566', '1997_61', '1997_617', '1997_730', '1997_795', '1997_803', '1997_880', '1997_92', '1997_924', '1998_1160', '1998_218', '1998_395', '1998_475', '1998_879', '1998_997', '1999_1072', '1999_1089', '1999_1090', '1999_119', '1999_152', '1999_186', '1999_443', '1999_444', '1999_489', '1999_545', '1999_65', '1999_686', '1999_76', '1999_947', '2000_1066', '2000_1155', '2000_1195', '2000_1504', '2000_1521', '2000_208', '2000_259', '2000_752', '2000_811', '2000_853', '2000_877', '2001_1030', '2001_1049', '2001_1112', '2001_1159', '2001_121', '2001_265', '2001_557', '2001_574', '2001_598', '2001_791', '2001_792', '2001_851', '2001_872', '2001_990', '2002_11', '2002_1147', '2002_1298', '2002_1302', '2002_1308', '2002_181', '2002_234', '2002_245', '2002_403', '2002_41', '2002_414', '2002_44', '2002_48', '2002_505', '2002_621', '2002_698', '2002_708', '2002_71', '2002_789', '2002_968', '2003_1073', '2003_1316', '2003_150', '2003_266', '2003_429', '2003_45', '2003_533', '2003_557', '2003_643', '2003_713', '2003_724', '2003_799', '2003_831', '2003_957', '2003_994', '2004_1029', '2004_1107', '2004_1171', '2004_170', '2004_184', '2004_332', '2004_470', '2004_606', '2004_854', '2004_928', '2005_106', '2005_285', '2005_41', '2005_415', '2005_417', '2005_469', '2005_570', '2005_887', '2006_1046', '2006_1050', '2006_1124', '2006_1211', '2006_287', '2006_447', '2006_468', '2006_469', '2006_584', '2006_781', '2006_824', '2006_88', '2006_975', '2006_981', '2007_1086', '2007_1220', '2007_1261', '2007_1356', '2007_137', '2007_1407', '2007_256', '2007_260', '2007_360', '2007_397', '2007_409', '2007_410', '2007_44', '2007_460', '2007_5', '2007_540', '2007_565', '2007_637', '2007_657', '2007_753', '2007_792', '2007_916', '2007_930', '2008_1027', '2008_1430', '2008_1459', '2008_1550', '2008_1941', '2008_195', '2008_2053', '2008_2086', '2008_2261', '2008_2447', '2008_2456', '2008_2469', '2008_2542', '2008_26', '2008_268', '2008_2686', '2008_369', '2008_408', '2008_433', '2008_63', '2009_1018', '2009_1095', '2009_1623', '2009_163', '2009_1684', '2009_1693', '2009_1794', '2009_1858', '2009_2122', '2009_217', '2009_221', '2009_293', '2009_299', '2009_353', '2009_391', '2009_542', '2009_619', '2009_762', '2009_961', '2010_1020', '2010_1172', '2010_1250', '2010_1266', '2010_334', '2010_436', '2010_505', '2010_523', '2010_549', '2010_6', '2010_64', '2010_696', '2010_72', '2010_890', '2011_103', '2011_1058', '2011_130', '2011_254', '2011_397', '2011_474', '2011_476', '2011_551', '2011_623', '2011_689', '2011_802', '2011_822', '2011_93', '2011_931', '2011_99', '2012_142', '2012_785', '2012_795', '2013_1008', '2013_127', '2013_136', '2013_493', '2013_61', '2013_761', '2013_780', '2013_982', '2014_162', '2014_331', '2014_416', '2014_672', '2014_800', '2015_259', '2015_314', '2015_327', '2015_442', '2015_55', '2015_605', '2015_801', '2016_122', '2016_172', '2016_306', '2016_360', '2016_393', '2016_495', '2016_568', '2016_591', '2016_595', '2016_660', '2016_692', '2016_76', '2017_112', '2017_115', '2017_137', '2017_164', '2017_224', '2017_297', '2017_323', '2017_461', '2017_573', '2018_1037', '2018_238', '2018_245', '2018_35', '2018_406', '2018_57', '2018_617', '2018_625', '2018_701', '2019_1089', '2019_114', '2019_255', '2019_408', '2019_419', '2019_459', '2019_484', '2019_485', '2019_527', '2019_55', '2019_675', '2019_785', '2019_786', '2019_805', '2019_946', '2019_999', '2020_182', '2020_313'])
EXPECTED_RECOVERED={'Train/train.json':{'sha256':'49bfc5f4d441833e2468e4c280c390ce361e2bfaffd5fbf1ad54157cddda164c','bytes':611284455},'Train/dev.json':{'sha256':'a071b080d3ffcd9789009cf46f6e4f1008142de0f44db9b97df9b64b6ee201c9','bytes':22387465},'Test/test.json':{'sha256':'88913c10c288a6f77a030d0391904d40b645fc242f2774f0ef8216e9f5afa792','bytes':35050736}}
recovered=[]
for p in Path('/kaggle/input').rglob('RECOVERY_MANIFEST.json'):
    try:
        if json.loads(p.read_text()).get('files')==EXPECTED_RECOVERED:recovered.append(p)
    except (ValueError,OSError):pass
assert len(recovered)==1,'Corrected recovered data unavailable or ambiguous.'
DATA=recovered[0].parent
for relative,wanted in EXPECTED_RECOVERED.items():
    assert (DATA/relative).stat().st_size==wanted['bytes'] and sha(DATA/relative)==wanted['sha256']
bundles=[]
for p in Path('/kaggle/input').rglob('INPUT_MANIFEST.json'):
    try:
        if json.loads(p.read_text()).get('family')=='b1':bundles.append(p.parent)
    except (ValueError,OSError):pass
assert len(bundles)==1,'Attach exact own Phase1 B1 input.'
BUNDLE=bundles[0]
assert sha(BUNDLE/'checkpoint.pt')=='d93a2f1796e90b9fdbe4a7b542f76eb6065dab619feed4a1ef5a9952e2376a1f'
assert sha(BUNDLE/'tokenizer.json')=='5c0bd30290e0bd6919253b1ff864f52053924808aabd33482df92cc4bd471c8e'
assert shutil.disk_usage(PERSIST).free>8*1024**3,'Need8GiB for best/latest states and exports.'
atomic_json(PERSIST/'INPUT_PROVENANCE.json',dict(data=EXPECTED_RECOVERED,bundle=str(BUNDLE),parent_sha256=sha(BUNDLE/'checkpoint.pt')))
CONTROL={'segment': 1, 'resume_dataset': None, 'expected_resume_manifest': None}
RESUME_ROOTS={}
if CONTROL.get('resume_dataset'):
    candidates=[]
    for p in Path('/kaggle/input').rglob('RESUME_MANIFEST.json'):
        if sha(p)==CONTROL['expected_resume_manifest']:candidates.append(p)
    assert len(candidates)==1,'Exact verified previous-segment resume manifest required.'
    manifest_path=candidates[0];resume=json.loads(manifest_path.read_text())
    for job in resume['jobs']:
        dest=WORK/'resume'/job['name'];dest.mkdir(parents=True,exist_ok=True)
        for name,record in job['files'].items():
            source=manifest_path.parent/record['filename']
            assert source.stat().st_size==record['bytes'] and sha(source)==record['sha256'],'Resume file mismatch'
            target=dest/name
            if not target.exists():target.symlink_to(source)
        RESUME_ROOTS[job['name']]=str(dest)
print('B1_STUDY_INPUTS_VERIFIED',{'bundle':str(BUNDLE),'data':str(DATA)})


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2,'Two isolated GPUs are required for this comparison.'
print('ALLOCATED_GPUS',[(i,torch.cuda.get_device_name(i)) for i in range(2)])
BASE=dict(seed=781,max_epochs=4,accumulation=8,warmup_updates=100,patience=5,backbone_lr=1e-5,head_lr=3e-4,
          encoder_depth=6,expand_ffn=False,chunk_batch=4,segment=CONTROL['segment'],segment_hours=3.,resume_root=None,
          bundle=str(BUNDLE),data_root=str(DATA),fixed_holdout=str(FIXED),session_end_unix=SESSION_END)
definitions=[dict(name='B1_FullData_Reference',representation='reference',head_mode='reference',max_tokens=256,chunks_per_case=8,overlap=0,window_policy='uniform'),
             dict(name='B1_DocumentHierarchy',representation='windows',head_mode='transformer_gru',max_tokens=512,chunks_per_case=16,overlap=100,window_policy='tail')]
JOBS=[]
for definition in definitions:
    config=dict(BASE,**definition)
    config.update(run_root=str(PERSIST/'runs'/config['name']),work_root=str(WORK/config['name']))
    config['resume_root']=RESUME_ROOTS.get(config['name'])
    JOBS.append(config)
atomic_json(PERSIST/'RUN_SCHEDULE.json',JOBS)
from b1_study.run import launch
RESULTS=launch(JOBS,PERSIST)
print('SEGMENT_FINISHED',[(r['name'],r['state'],r.get('optimizer_updates'),r.get('completed_epochs')) for r in RESULTS])
